# Cycle 3 — Modelling (Chronological Split)

The processed injuries file dropped `start_year`. We re-load the raw file, redo the same preprocessing steps, but **keep `start_year`** so we can split by season. Latest seasons go to test, earlier seasons train.

In [3]:
import sys, os                                
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')             

from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

_here = os.getcwd()                                       
while not os.path.isdir(os.path.join(_here, 'data')):     
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                             

from config import Paths, ensure_dirs
ensure_dirs()  


## Re-preprocess raw injuries keeping(start_year)

Reloads the raw injuries CSV and repeats every transformation from `cycle3_preprocessing_injuries.ipynb` (target encoding, ordinal mappings, BMI, leakage drops, NaN fills) — but **omits the drop of `start_year`**, which the original preprocessing discarded.

In [4]:
raw = pd.read_csv(str(Paths.PLAYER_INJURIES_RAW))          # load raw — processed CSV lacks start_year
print(f'Raw rows: {len(raw)}')

THRESHOLD = 28                                             # 28 days ≈ one month — meaningful absence
raw['High_Injury'] = (raw['season_days_injured'] >= THRESHOLD).astype(int)  # binary target

# Recreate engineered features from the original preprocessing notebook
wr_map  = {'Low':1,'Medium':2,'High':3}
pos_map = {'GK':1,'DF':2,'MF':3,'FW':4}
if 'work_rate' in raw.columns:
    raw['work_rate_numeric'] = raw['work_rate'].astype(str).str.split('/').str[0].str.strip().map(wr_map).fillna(2)
if 'position' in raw.columns:
    raw['position_numeric'] = raw['position'].astype(str).str[:2].map(pos_map).fillna(3)
if 'height_cm' in raw.columns and 'weight_kg' in raw.columns:
    raw['bmi'] = raw['weight_kg'] / (raw['height_cm']/100.0)**2  # Body Mass Index proxy for physique

# Drop leakage columns but KEEP start_year for the temporal split
drop_cols = ['p_id2','dob','nationality','work_rate','position',
             'season_days_injured','total_days_injured',
             'season_minutes_played','season_games_played','season_matches_in_squad',
             'total_minutes_played','total_games_played']
df = raw.drop(columns=[c for c in drop_cols if c in raw.columns])

# Fill missing history columns with zeros — first-season players have no prior record
history_cols = ['cumulative_minutes_played','cumulative_games_played',
                'minutes_per_game_prev_seasons','avg_days_injured_prev_seasons',
                'avg_games_per_season_prev_seasons','significant_injury_prev_season',
                'cumulative_days_injured','season_days_injured_prev_season']
history_cols = [c for c in history_cols if c in df.columns]
df[history_cols] = df[history_cols].fillna(0)              # 0 = 'no recorded history'

df = df.dropna().reset_index(drop=True)
print(f'After preprocessing: {len(df)} rows | start_year range: {df["start_year"].min()}-{df["start_year"].max()}')
print(f'Class balance: High={df["High_Injury"].mean()*100:.1f}%')


Raw rows: 1301
After preprocessing: 1206 rows | start_year range: 2016-2020
Class balance: High=70.1%


## Chronological split by start_year

Arranging seasons in ascending order (from oldest to newest). We train the model on the first 80% of the data and test it on the final 20% (the most recent season).

In [5]:
# Sort by season so row order is strictly temporal
df = df.sort_values('start_year').reset_index(drop=True)
split_idx = int(len(df) * 0.8)                            # 80/20 target split
boundary_year = df['start_year'].iloc[split_idx]          # season at the 80th-percentile row

train_df = df[df['start_year'] < boundary_year]           # all earlier seasons → train
test_df  = df[df['start_year'] >= boundary_year]          # boundary season and later → test
# Fallback: if year boundary creates a heavily skewed split, use row-index split instead
if len(test_df) < 0.1*len(df) or len(test_df) > 0.4*len(df):
    train_df = df.iloc[:split_idx]
    test_df  = df.iloc[split_idx:]

drop_for_X = ['High_Injury','start_year']                 # remove target and temporal key from features
X_train = train_df.drop(columns=drop_for_X)
y_train = train_df['High_Injury']
X_test  = test_df.drop(columns=drop_for_X)
y_test  = test_df['High_Injury']

# Align column order with the saved feature list so API input shape is consistent
import joblib, os
feature_cols_path = str(Paths.C3_FEATURES)
if os.path.exists(feature_cols_path):
    expected = joblib.load(feature_cols_path)
    extras = [c for c in X_train.columns if c not in expected]
    if extras: X_train = X_train.drop(columns=extras); X_test = X_test.drop(columns=extras)
    missing = [c for c in expected if c not in X_train.columns]
    for m in missing: X_train[m]=0.0; X_test[m]=0.0      # pad any missing column with 0
    X_train = X_train[expected]
    X_test  = X_test[expected]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)                 # fit on train only — avoids test leakage
X_test_s  = scaler.transform(X_test)

spw = (y_train==0).sum()/(y_train==1).sum()               # Low-to-High ratio for XGBoost/LightGBM

print(f'Train years: {train_df["start_year"].min()} -> {train_df["start_year"].max()}')
print(f'Test  years: {test_df["start_year"].min()} -> {test_df["start_year"].max()}')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train high-injury rate: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%')
print(f'Features ({len(X_train.columns)}): {list(X_train.columns)}')


Train years: 2016 -> 2019
Test  years: 2020 -> 2020
Train: 956 | Test: 250
Train high-injury rate: 72.3% | Test: 62.0%
Features (17): ['height_cm', 'weight_kg', 'pace', 'physic', 'fifa_rating', 'age', 'cumulative_minutes_played', 'cumulative_games_played', 'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons', 'avg_games_per_season_prev_seasons', 'bmi', 'work_rate_numeric', 'position_numeric', 'significant_injury_prev_season', 'cumulative_days_injured', 'season_days_injured_prev_season']


### Dummy classifier baseline

In [6]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)  # always predicts High Injury (majority)
dummy.fit(X_train_s, y_train)
y_pred_d = dummy.predict(X_test_s); y_prob_d = dummy.predict_proba(X_test_s)[:,1]
print(f'DUMMY -- Acc {accuracy_score(y_test,y_pred_d)*100:.2f}% | AUC {roc_auc_score(y_test,y_prob_d):.4f}')
print(classification_report(y_test, y_pred_d, target_names=['Low Injury','High Injury']))


DUMMY -- Acc 62.00% | AUC 0.5000
              precision    recall  f1-score   support

  Low Injury       0.00      0.00      0.00        95
 High Injury       0.62      1.00      0.77       155

    accuracy                           0.62       250
   macro avg       0.31      0.50      0.38       250
weighted avg       0.38      0.62      0.47       250



### Logistic Regression

In [7]:
lr = LogisticRegression(class_weight='balanced',   # up-weight Low Injury to counter 70/30 imbalance
                         max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s); y_prob_lr = lr.predict_proba(X_test_s)[:,1]
print(f'LOGREG -- Acc {accuracy_score(y_test,y_pred_lr)*100:.2f}% | AUC {roc_auc_score(y_test,y_prob_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Low Injury','High Injury']))


LOGREG -- Acc 54.40% | AUC 0.6322
              precision    recall  f1-score   support

  Low Injury       0.43      0.66      0.53        95
 High Injury       0.70      0.47      0.56       155

    accuracy                           0.54       250
   macro avg       0.56      0.57      0.54       250
weighted avg       0.60      0.54      0.55       250



### Random Forest

Trains a 200-tree Random Forest with `class_weight='balanced'` on the scaled chronological training partition and evaluates on the test season.

In [8]:
rf = RandomForestClassifier(n_estimators=200,            # 200 trees for stable probability estimates
                             class_weight='balanced',    # adjust sample weights for class imbalance
                             random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s); y_prob_rf = rf.predict_proba(X_test_s)[:,1]
print(f'RF -- Acc {accuracy_score(y_test,y_pred_rf)*100:.2f}% | AUC {roc_auc_score(y_test,y_prob_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Low Injury','High Injury']))


RF -- Acc 65.20% | AUC 0.6428
              precision    recall  f1-score   support

  Low Injury       0.62      0.21      0.31        95
 High Injury       0.66      0.92      0.77       155

    accuracy                           0.65       250
   macro avg       0.64      0.57      0.54       250
weighted avg       0.64      0.65      0.60       250



### XGBoost

Trains an XGBoost classifier with `scale_pos_weight=spw` (computed from the training-partition class ratio) and `eval_metric='auc'` on the scaled chronological split. Evaluates on the test season.

In [9]:
xgb = XGBClassifier(scale_pos_weight=spw,            # weight minority class (Low Injury) inversely
                     random_state=42, eval_metric='auc', verbosity=0)
xgb.fit(X_train_s, y_train)
y_pred_xgb = xgb.predict(X_test_s); y_prob_xgb = xgb.predict_proba(X_test_s)[:,1]
print(f'XGB -- Acc {accuracy_score(y_test,y_pred_xgb)*100:.2f}% | AUC {roc_auc_score(y_test,y_prob_xgb):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=['Low Injury','High Injury']))


XGB -- Acc 64.80% | AUC 0.6539
              precision    recall  f1-score   support

  Low Injury       0.55      0.43      0.48        95
 High Injury       0.69      0.78      0.73       155

    accuracy                           0.65       250
   macro avg       0.62      0.61      0.61       250
weighted avg       0.64      0.65      0.64       250



### LightGBM

Trains a 100-tree LightGBM classifier with `scale_pos_weight=spw` and `verbose=-1` on the scaled chronological training partition. Evaluates on the test season.

In [10]:
lgb = LGBMClassifier(scale_pos_weight=spw,    # same imbalance correction as XGBoost
                      n_estimators=100, random_state=42, verbose=-1)
lgb.fit(X_train_s, y_train)
y_pred_lgb = lgb.predict(X_test_s); y_prob_lgb = lgb.predict_proba(X_test_s)[:,1]
print(f'LGB -- Acc {accuracy_score(y_test,y_pred_lgb)*100:.2f}% | AUC {roc_auc_score(y_test,y_prob_lgb):.4f}')
print(classification_report(y_test, y_pred_lgb, target_names=['Low Injury','High Injury']))


LGB -- Acc 65.60% | AUC 0.6568
              precision    recall  f1-score   support

  Low Injury       0.56      0.47      0.51        95
 High Injury       0.70      0.77      0.73       155

    accuracy                           0.66       250
   macro avg       0.63      0.62      0.62       250
weighted avg       0.65      0.66      0.65       250



## Side-by-side summary

Random-split AUCs (from `notebooks/cycle3_modelling.ipynb`): Dummy 0.500, LR 0.6220, RF 0.5916, XGB 0.6179.

In [11]:
results = pd.DataFrame([
    {'Model':'Dummy',               'Chrono AUC':roc_auc_score(y_test,y_prob_d),     'Random AUC (orig)':0.5000},
    {'Model':'Logistic Regression', 'Chrono AUC':roc_auc_score(y_test,y_prob_lr),    'Random AUC (orig)':0.6220},
    {'Model':'Random Forest',       'Chrono AUC':roc_auc_score(y_test,y_prob_rf),    'Random AUC (orig)':0.5916},
    {'Model':'XGBoost',             'Chrono AUC':roc_auc_score(y_test,y_prob_xgb),   'Random AUC (orig)':0.6179},
    {'Model':'LightGBM',            'Chrono AUC':roc_auc_score(y_test,y_prob_lgb),   'Random AUC (orig)':0.6355},
])
results['Delta']      = (results['Chrono AUC']-results['Random AUC (orig)']).round(4)  # positive = chrono harder; negative = random was inflated by future leakage
results['Chrono AUC'] = results['Chrono AUC'].round(4)
print(results.to_string(index=False))
print()
print('Interpretation:')
print('  Positive Delta = chronological split is harder than random split')
print('  Negative Delta = chronological split is easier (model learns seasonal patterns)')


              Model  Chrono AUC  Random AUC (orig)  Delta
              Dummy      0.5000             0.5000 0.0000
Logistic Regression      0.6322             0.6220 0.0102
      Random Forest      0.6428             0.5916 0.0512
            XGBoost      0.6539             0.6179 0.0360
           LightGBM      0.6568             0.6355 0.0213

Interpretation:
  Positive Delta = chronological split is harder than random split
  Negative Delta = chronological split is easier (model learns seasonal patterns)


- All real models improve under the chronologicla split (positive delta). Indicating that perfomance is not inflated by data leakage from random splitting. 
- **LIGHTGBM** reamins the best model (AUC 0.6568), confirming its robustness even under a more realistic time-based evaluation
- XGBoost and Random Forest show the largest gains (+0.0360 and +0.0512), suggesting they benefit from temporal structure and consistent patterns across seasons. 
- Logistic Regression shows only a small improvement (+0.0102), indicating it captures more stable, linear relationships that are less dependent on time
- The Dummy model remains unchanged (0.5), as expected, confirming no effect from split strategy
- This suggests the chronological split is more realistic for depolyment, as it mimics predicting future seasons from past data